In [1]:
import os
import re
out = os.listdir("/")
target_dir = "/content"
pattern = r"^content_[\w-]+$"

result = [x for x in out if re.match(pattern, x)][0]
if result:
    target_dir = result
target_dir

'content_4c2e7447-7da4-4d32-86cc-b50527d5cf77'

In [2]:
os.listdir(f"/{target_dir}")

['RCLONE_TEST',
 'stage2_train.jsonl',
 'stage3_train.jsonl',
 'stage1_train.jsonl']

In [3]:
!pip install unsloth trl datasets wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 8.8 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.3 MB/s eta 0:00:00
 

In [4]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments, TrainerCallback
from datasets import load_dataset
from google.colab import drive
import os
import glob
import matplotlib.pyplot as plt

# Change this to your actual folder path
CHECKPOINT_DIR = f"/{target_dir}/cp"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2. MODEL CONFIGURATION
model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B"
max_seq_length = 8192
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
)

# 3. LORA ADAPTERS
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. PROMPT TEMPLATE (Same as before)
train_prompt_style = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a Python Competitive Programming Expert. Follow this structure exactly:
1. Provide metadata in <tags> [DIFFICULTY], [TOPICS] </tags>.
2. Reason step-by-step inside <think> </think>.
3. Provide final runnable Python code inside <code> </code>.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{instruction}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
{output}<|eot_id|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = [train_prompt_style.format(instruction=i, output=o) for i, o in zip(instructions, outputs)]
    return { "text" : texts }

# 5. DATA PREPARATION
dataset = load_dataset("json", data_files=f"/{target_dir}/stage1_train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 6. TRAINING ARGUMENTS (NO W&B)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,
        max_steps = 2250,      # 1.5 Epochs
        learning_rate = 2e-4,
        fp16 = True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = CHECKPOINT_DIR,
        save_strategy = "steps",
        save_steps = 250,
        save_total_limit = 2,
        logging_steps = 10,
        report_to = "none",    # <--- W&B DISABLED
    ),
)

# 7. SMART RESUME LOGIC
print("🚀 Starting/Resuming Stage 1 Foundation...")
checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*"))

if checkpoints:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"Found existing checkpoint: {latest_checkpoint}. Resuming...")
    trainer.train(resume_from_checkpoint = latest_checkpoint)
else:
    print("No valid checkpoints found. Starting from scratch.")
    trainer.train()

# 8. FINAL SAVE
model.save_pretrained_merged(f"/{target_dir}/DeepSeek-R1-CP-Python-Stage1", tokenizer, save_method = "merged_16bit")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:31<00:00, 31.89s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:33<00:00, 33.58s/it]


Unsloth: Merge process complete. Saved to `/content_4c2e7447-7da4-4d32-86cc-b50527d5cf77/DeepSeek-R1-CP-Python-Stage1`
